# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² tabular dataset of 77 cancer survivors with second primary colorectal cancer, using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. You'll see examples of working with Croissant's record sets, fields, and columns referenced by their `@id` according to the Croissant schema.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- Title: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset via the provided URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("\033[1mDataset loaded.\033[0m")
print("\nTitle:", metadata.name)
print("Description:", metadata.description)
print("Published:", getattr(metadata, 'datePublished', 'N/A'))
print("Identifier:", getattr(metadata, 'identifier', 'N/A'))
print("License:", getattr(metadata, 'license', 'N/A'))

## 2. Data Overview
Inspect available record sets, fields, and their respective `@id`s (identifiers).
We'll print a summary of record sets and their main fields, referencing them strictly by their `@id` as required.

In [ ]:
# List all record sets with their @id and field @ids

print("\033[1mAvailable Record Sets:\033[0m")
record_sets = list(dataset.record_sets())
for recset in record_sets:
    print(f"- Record Set: {recset['@id']}")
    if 'field' in recset:
        fields = recset['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("    Fields:")
        for field in fields:
            print(f"      - {field['@id']}")
    else:
        print("    (No fields found)")

# For demonstration, preview the first record from each record set (by @id)
print("\n\033[1mSample Record from Each Record Set:\033[0m")
for recset in record_sets:
    recset_id = recset['@id']
    try:
        record_iterator = dataset.records(record_set=recset_id)
        first_record = next(record_iterator, None)
        print(f"- {recset_id}:\n  {first_record}\n")
    except Exception as e:
        print(f"- {recset_id}: Error loading record ({e})\n")

## 3. Data Extraction
Now, let's load all the records from each record set into a Pandas DataFrame for further analysis. We'll use their `@id`s as the keys, as per instructions.

**Tips:**
- Replace `<record_set_id>` and field names with the actual `@id` values identified above when operating on specific data.

In [ ]:
# Extract records from all record sets and collect DataFrames by their @id
dataframes = {}
for recset in record_sets:
    recset_id = recset['@id']
    all_records = list(dataset.records(record_set=recset_id))
    if all_records:
        dataframes[recset_id] = pd.DataFrame(all_records)
        print(f"Loaded {len(all_records)} records for record set: {recset_id}")
    else:
        print(f"No records for record set: {recset_id}")

# --- Choose main tabular data record set for exploration ---
# Let's pick the largest set or main data table for demonstration.
chosen_record_set_id = None
max_rows = 0
for recset_id, df in dataframes.items():
    if len(df) > max_rows:
        max_rows = len(df)
        chosen_record_set_id = recset_id

print("\n\033[1mColumns in main data record set:\033[0m")
if chosen_record_set_id:
    print(f"Record set @id: {chosen_record_set_id}")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())
else:
    print("No record set with data found.")

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate some common processing steps. Please refer to the field `@id`s and column names from above for actual analysis.
- **Filtering:** Filter records based on a numeric field exceeding a threshold.
- **Normalization:** Z-normalize a chosen numeric field.
- **Grouping/Aggregation:** Group by a key attribute if available.

_If the dataset contains age, we'll use it as an example; otherwise, adapt to a different numeric field in the DataFrame._

In [ ]:
df = dataframes[chosen_record_set_id]

# Identify a numeric field by @id (e.g. 'age' or similar, adapt as needed)
numeric_field_cands = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
print(f"Available numeric fields: {numeric_field_cands}")
if numeric_field_cands:
    numeric_field_id = numeric_field_cands[0]  # Use first available as an example
else:
    numeric_field_id = df.columns[0]  # fallback if no numeric types detected

print(f"\nSelected numeric field for EDA: {numeric_field_id}\n")

# Simple threshold filtering
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std()!=0 else 1)
)
print(f"Normalized {numeric_field_id} (z-score):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if available
group_field_cands = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) or df[col].dtype=='category']
group_field_id = None
for field in group_field_cands:
    if df[field].nunique() < len(df) // 2 and df[field].nunique()>1:
        group_field_id = field
        break
# Grouping and aggregation
if group_field_id:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
    display(grouped.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (and optionally categories), using standard libraries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], kde=True, bins=20, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If possible, stratified boxplot by first available grouping field
if group_field_id:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
- We used `mlcroissant` to read metadata and records from a Croissant-annotated clinical dataset, referencing all data entities by their official `@id`s.
- Tabular data were loaded to Pandas DataFrames for further processing.
- We identified numeric and categorical fields for demonstration and performed filtering, normalization, and grouping operations.
- Quick visualizations helped summarize variable distributions and highlight potential subgroup differences in the data.

Refer to Croissant and the dataset source for details on schema semantics, provenance, and recommended analysis standards!